In [1]:
import kglab
import pickle
import rdflib 
import re
import torch
import json 
import copy 
import urllib.parse
import time

import networkx as nx
import pandas as pd
import numpy as np

from tqdm.notebook import tqdm
from collections import Counter, defaultdict
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity

C:\Users\roans\anaconda3.0\envs\transformers\lib\site-packages\kglab\util.py:35: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore  # pylint: disable=E0401


In [2]:
df = pd.read_excel(f"../outputs/clean_outputs/triples_coalesced.xlsx").drop("Unnamed: 0", axis=1)

In [3]:
df.head()

,id,company,job title,text,triples_qwen_structured,triples_qwen_semi-structured,triples_qwen_unstructured,triples_gemma_structured,triples_gemma_semi-structured,triples_gemma_unstructured,triples_llama_structured,triples_llama_semi-structured,triples_llama_unstructured
0,1527392,Jobindex,"IT-administrator – få indflydelse på et setup,...",Vil du ind i en virksomhed i rivende udvikling...,"[('IT Administrator', 'REQUIRES_SKILL', 'setup...","[('IT Administrator', 'REQUIRES_SKILL', 'netwo...","[('IT Administrator', 'require', 'technical_se...","[('IT Administrator', 'INVOLVES_TASK', 'Mainta...","[('IT Administrator', 'REQUIRES_SKILL', 'Syste...",[('IT administrator • influence a setup that d...,"[('IT Administrator', 'REQUIRES_SKILL', 'setup...","[('IT Administrator', 'REQUIRES_SKILL', 'Docks...","[('IT Administrator', 'have', 'influence on a ..."
1,1527395,Jobindex,"IT-administrator – få indflydelse på et setup,...","IT-administrator – få indflydelse på et setup,...","[('IT Administrator', 'REQUIRES_SKILL', 'setup...","[('IT Administrator', 'REQUIRES_SKILL', 'netwo...","[('IT Administrator', 'require', 'technical_se...","[('IT Administrator', 'REQUIRES_SKILL', 'IT Ad...","[('IT Administrator', 'REQUIRES_SKILL', 'Syste...",[('IT administrator • influence a setup that d...,"[('IT Administrator', 'REQUIRES_SKILL', 'Docks...",[('IT administrator • influence a setup that d...,"[('IT Administrator', 'have', 'influence on a ..."
2,1527397,Aqua d'Or Mineral Water A/S,SQE Manager,For jobsøgere For arbejdsgivere Aqua d'Or Mi...,"[('SQE Manager', 'REQUIRES_SKILL', 'Root Cause...","[('SQE Manager', 'REQUIRES_SKILL', 'Quality Ma...","[('SQE Manager', 'require', 'quality assurance...","[('SQE Manager', 'REQUIRES_SKILL', 'Quality As...","[('SQE Manager', 'REQUIRES_SKILL', 'Quality As...","[('SQE Manager', 'title', 'SQE Manager'), ('SQ...","[('SQE Manager', 'REQUIRES_QUALITY', 'detailed...","[('SQE Manager', 'REQUIRES_SKILL', 'DevOps Eng...","[('SQE Manager', 'require', 'English language ..."
3,1527417,Klimabrands,Kundeservice / teknisk support,For jobsøgere For arbejdsgivere mailto:job@k...,"[('Customer Service / Technical Support', 'REQ...","[('Customer Service / Technical Support', 'REQ...","[('Customer Service / Technical Support', 'req...","[('Customer Service / Technical Support', 'REQ...","[('Customer Service / Technical Support', 'REQ...","[('Customer Service / Technical Support', 'tit...","[('Customer service', 'REQUIRES_QUALITY', 'det...","[('Customer service', 'REQUIRES_SKILL', 'Techn...","[('Customer service', 'have', 'a technical sup..."
4,1527440,Scan Studio ApS,Retail designer med teknikken på plads,For jobsøgere For arbejdsgivere Scan Studio ...,"[('Retail Designer', 'REQUIRES_SKILL', 'Techni...","[('Retail Designer', 'REQUIRES_SKILL', 'Design...","[('Retail Designer', 'require', 'technical ski...","[('Retail designer', 'REQUIRES_SKILL', 'the te...","[('Retail designer', 'REQUIRES_SKILL', 'the te...",[('Retail designer with the technique in place...,[('Retail designer with the technique in place...,"[('Danish designer', 'REQUIRES_QUALITY', 'SKIL...",[('Retail designer requires technical competen...


In [4]:
df_isco = pd.read_excel("../outputs/raw_outputs/ISCO_ESCO_triples.xlsx")
df_isco[["ISCO qwen structured", "ESCO qwen structured"]].head()

,ISCO qwen structured,ESCO qwen structured
0,"[(""IT-administrator"", ""has_isco"", ""2521"")]","[(""IT-administrator"", ""has_esco"", ""011151"")]"
1,"[(""SQE Manager"", ""has_isco"", ""7543"")]","[(""SQE Manager"", ""has_esco"", ""008265""), (""SQE ..."
2,"[(""Kundeservice / teknisk support"", ""has_isco""...","[(""technical communication"", ""has_esco"", ""0065..."
3,[],[]
4,[],"[(""apply supports for spinal adjustment"", ""has..."


In [6]:
iscos = pd.read_csv("../pipeline/data_files/isco.csv", encoding="cp850")
iscos = iscos[iscos["ISCO_version"] == "ISCO-08"]

iscos.head()

,ISCO_version,major,major_label,sub_major,sub_major_label,minor,minor_label,unit,description
0,ISCO-08,1,Managers,11.0,"Chief Executives, Senior Officials and Legisla...",111,Legislators and Senior Officials,1111,Legislators
1,ISCO-08,1,Managers,11.0,"Chief Executives, Senior Officials and Legisla...",111,Legislators and Senior Officials,1112,Senior Government Officials
2,ISCO-08,1,Managers,11.0,"Chief Executives, Senior Officials and Legisla...",111,Legislators and Senior Officials,1113,Traditional Chiefs and Heads of Villages
3,ISCO-08,1,Managers,11.0,"Chief Executives, Senior Officials and Legisla...",111,Legislators and Senior Officials,1114,Senior Officials of Special-interest Organizat...
4,ISCO-08,1,Managers,11.0,"Chief Executives, Senior Officials and Legisla...",112,Managing Directors and Chief Executives,1120,Managing Directors and Chief Executives


In [7]:
namespaces = {
    "jip" : "http://company.com/property/",
    "jid" : "http://company.com/ontology/",
    "jie" : "http://company.com/entity/",
    "dbp": "http://dbpedia.org/property/",
    "dbo": "http://dbpedia.org/ontology/",
    "dbr": "http://dbpedia.org/resource/",
    "owl": "http://www.w3.org/2002/07/owl#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "foaf": "http://xmlns.com/foaf/0.1/"
}

kg = kglab.KnowledgeGraph(
    name = "Jobindex KG",
    namespaces = namespaces,
    base_uri = "https://www.example.com/entity/"
)

In [8]:
# Define ontology lay-out

# Classes

# Candidate is a class, and a candidate is a person
kg.add(kg.get_ns("jid").Candidate, kg.get_ns("rdf").type, kg.get_ns("owl").Class)
kg.add(kg.get_ns("jid").Candidate, kg.get_ns("rdfs").subClassOf, kg.get_ns("foaf").Person)

# Function is a class, and is equivalent to employment
kg.add(kg.get_ns("jid").Position, kg.get_ns("rdf").type, kg.get_ns("owl").Class)

# Add educations and their ordering
edus = ["Doctorate", "PhD", "Masters_degree", "Bachelors_degree",
        "College_degree", "Upper_high_school_diploma", "Trade_school_diploma",
        "Primary_middle_school_only"]

for i, edu in enumerate(edus):
    kg.add(eval(f"kg.get_ns('jie').{edu}"), kg.get_ns("rdf").type, kg.get_ns("jid").Education)
    
    for edu2 in edus[i+1:]:
        kg.add(eval(f"kg.get_ns('jie').{edu}"), kg.get_ns("jid").supersedes, eval(f"kg.get_ns('jie').{edu2}"))

# Company is a class, it offers functions, and candidates work there
kg.add(kg.get_ns("jid").Company, kg.get_ns("rdf").type, kg.get_ns("owl").Class)
kg.add(kg.get_ns("jid").Company, kg.get_ns("jip").offers_position, kg.get_ns("jid").Position)
kg.add(kg.get_ns("jid").Candidate, kg.get_ns("jip").has_worked_at, kg.get_ns("jid").Company)
kg.add(kg.get_ns("jid").Candidate, kg.get_ns("jip").has_worked_position, kg.get_ns("jid").Position)

### SINCE ISCO IS SO IMPORTANT, MAYBE TURN THE LLM PART INTO A PIPELINE --> EXTRACT TRIPLES, LINK TRIPLES TO ISCO
# An ISCO code is a class, and all sub-codes fall under that class
kg.add(kg.get_ns("jid").ISCO_code, kg.get_ns("rdf").type, kg.get_ns("owl").Class)
kg.add(kg.get_ns("jid").ISCO_unit, kg.get_ns("rdfs").subClassOf, kg.get_ns("jid").ISCO_code)
kg.add(kg.get_ns("jid").ISCO_minor, kg.get_ns("rdfs").subClassOf, kg.get_ns("jid").ISCO_code)
kg.add(kg.get_ns("jid").ISCO_sub_major, kg.get_ns("rdfs").subClassOf, kg.get_ns("jid").ISCO_code)
kg.add(kg.get_ns("jid").ISCO_major, kg.get_ns("rdfs").subClassOf, kg.get_ns("jid").ISCO_code)

# Add skill, language, and license class
kg.add(kg.get_ns("jid").Skill, kg.get_ns("rdf").type, kg.get_ns("owl").Class)
kg.add(kg.get_ns("jid").Language, kg.get_ns("rdf").type, kg.get_ns("owl").Class)
kg.add(kg.get_ns("jid").Certificate, kg.get_ns("rdf").type, kg.get_ns("owl").Class)

# Isco levels
for row in iscos.itertuples():
    # Unit
    kg.add(eval(f"kg.get_ns('jie').isco{row[8]}"), kg.get_ns("rdf").type, kg.get_ns("jid").ISCO_unit)
    kg.add(eval(f"kg.get_ns('jie').isco{row[8]}"), kg.get_ns("jip").falls_under, eval(f"kg.get_ns('jie').isco{row[6]}"))
    kg.add(eval(f"kg.get_ns('jie').isco{row[8]}"), kg.get_ns("rdfs").comment, rdflib.Literal(row[9]))
    
    # Minor
    kg.add(eval(f"kg.get_ns('jie').isco{row[6]}"), kg.get_ns("rdf").type, kg.get_ns("jid").ISCO_minor)
    kg.add(eval(f"kg.get_ns('jie').isco{row[6]}"), kg.get_ns("jip").falls_under, eval(f"kg.get_ns('jie').isco{int(row[4])}"))
    kg.add(eval(f"kg.get_ns('jie').isco{row[6]}"), kg.get_ns("rdfs").comment, rdflib.Literal(row[7]))

    # Sub_major
    kg.add(eval(f"kg.get_ns('jie').isco{int(row[4])}"), kg.get_ns("rdf").type, kg.get_ns("jid").ISCO_sub_major)
    kg.add(eval(f"kg.get_ns('jie').isco{int(row[4])}"), kg.get_ns("jip").falls_under, eval(f"kg.get_ns('jie').isco{row[2]}"))
    kg.add(eval(f"kg.get_ns('jie').isco{int(row[4])}"), kg.get_ns("rdfs").comment, rdflib.Literal(row[5]))

    # Major
    kg.add(eval(f"kg.get_ns('jie').isco{row[2]}"), kg.get_ns("rdf").type, kg.get_ns("jid").ISCO_major)
    kg.add(eval(f"kg.get_ns('jie').isco{row[2]}"), kg.get_ns("rdfs").comment, rdflib.Literal(row[3]))
    
# Properties
kg.add(kg.get_ns("jip").offers_position, kg.get_ns("owl").inverseOf, kg.get_ns("jip").position_is_offered_by)

kg.add(kg.get_ns("jip").has_worked_at, kg.get_ns("owl").inverseOf, kg.get_ns("jip").has_employed)

kg.add(kg.get_ns("jip").supersedes, kg.get_ns("owl").inverseOf, kg.get_ns("jip").subsedes)

kg.add(kg.get_ns("jip").falls_under, kg.get_ns("owl").inverseOf, kg.get_ns("jip").encompasses)

# Falls under is transitive
kg.add(kg.get_ns("jip").falls_under, kg.get_ns("rdf").type, kg.get_ns("owl").TransitiveProperty)

In [9]:
measure = kglab.Measure()
measure.measure_graph(kg)

print("edges before inference", measure.get_edge_count())
print("nodes before inference", measure.get_node_count())

# Do all the nifty inference
kg.infer_owlrl_closure()

measure.measure_graph(kg)

print()
print("edges after inference", measure.get_edge_count())
print("nodes after inference", measure.get_node_count())

edges before inference 1900
nodes before inference 644

edges after inference 9559
nodes after inference 1282


In [11]:
with open("../outputs/clean_outputs/anon_cvs_coalesced.json", 'r', encoding="utf-8") as f:
    data = json.load(f)

In [12]:
def extract_triples(input_string):
    """
    Extracts only 3-element tuples, accepting single or double quotes.
    """

    input_string = str(input_string)
    processed_string = input_string.replace('*', '"')

    # Quoting Group (Q): This group (r'["\']') matches either a double quote OR a single quote.
    Q = r'["\']' 
        
    # Flexible Pattern (using f-string for clarity and Q definition):
    pattern = rf"""
        \(              # Match the literal opening parenthesis (
        ({Q}.*?{Q})     # Group 1: Capture the first quoted string
        ,\s* # Match comma, optional whitespace
        ({Q}.*?{Q})     # Group 2: Capture the second quoted string
        ,\s* # Match comma, optional whitespace
        ({Q}.*?{Q})     # Group 3: Capture the third quoted string
        \)              # Match the literal closing parenthesis )
    """
    
    # Use re.findall with re.VERBOSE for multiline pattern and re.DOTALL to match across newlines
    matches = re.findall(pattern, processed_string, re.VERBOSE | re.DOTALL)
    
    extracted_data = []
    for str1_quoted, str2_quoted, str3_quoted in matches:
        # Remove the surrounding quotes from each captured string
        # using the replace method, which handles both ' and "
        str1 = str1_quoted.strip().replace('"', '').replace("'", '')
        str2 = str2_quoted.strip().replace('"', '').replace("'", '')
        str3 = str3_quoted.strip().replace('"', '').replace("'", '')
        extracted_data.append((str1, str2, str3))
        
    return extracted_data

In [13]:
def clean_label(text):
    # 1. Replace all non-alphanumerics with underscores
    # Note: \w includes letters like Ä, ö, etc.
    s = re.sub(r'[^\w]', '_', str(text)).lower()
    
    # 2. Collapse multiple leading underscores into one
    s = re.sub(r'_+', '_', s)
    
    return urllib.parse.quote(s)

def add_triple(kg, s, p, o, literal=False):
    ns_jie = kg.get_ns('jie')
    ns_jip = kg.get_ns('jip')

    s, p, o = clean_label(s), clean_label(p), clean_label(o)

    # Do not store NaNs
    if any([pd.isna(i) for i in [s, p, o]]):
        return 1

    # Non-existent node
    if not literal and not len(o):
        return 1

    # Some nodes start with a digit, which is not allowed
    if not literal and o[0].isdigit():
        o = list(o)       
        o = "_" + o

        if len(o) == 2:
            o = "int_" + o

    if s[0].isdigit():
        s = "_" + s
            
    if p[0].isdigit():
        p = "_" + p 

    try:
        if not literal:         
            # Sometimes all cleaning leads to a string
            # that only contains underscores
            if set(o[1]) == {"_"}:
                return 1
                
            kg.add(getattr(ns_jie, s), 
                   getattr(ns_jip, p), 
                   getattr(ns_jie, o))
        else:
            # Replace nan with 0
            if o in ["nan", "none", ""] or pd.isna(o):
                o = 0
                
            kg.add(getattr(ns_jie, s), 
                   getattr(ns_jip, p),
                   rdflib.Literal(int(o), datatype=kg.get_ns("xsd").integer))

    except Exception as e:
        print(e)
        print(s, p, o)

    return 0

def prepare_node(text):
    if (type(text) == float and np.isnan(text)) or (not text):
        return "none"
    
    text = text.lower()
    
    # "separating" characters become underscores
    cleaned_text = re.sub(r'[ /\\.,:;()&-]+', '_', text)
    
    # All other non-whitespace characters get removed
    final_text = re.sub(r'[^a-zA-Z0-9_]', '', cleaned_text)

    if final_text in ["import", "global", "is", "as", "or", 
                      "and", "in", "for", "while", "lambda",
                      "return", "if", "else", "encode"]:
        final_text = "_" + final_text
    
    return final_text

country_map = {4: "denmark"}

education_map = {0: "Primary_middle_school_only",
                 1: "Trade_school_diploma",
                 3: "Upper_high_school_diploma",
                 6: "College_degree",
                 8: "Bachelors_degree",
                 9: "Masters_degree", 
                 12: "PhD"}

fluency_map = {0: "speaks_little", 
               1: "speaks_beginner", 
               2: "speaks_intermediate",
               3: "speaks_advanced",
               4: "speaks_fluent"}

language_map = {"aa": "Afar", "ab": "Abkhazian", "ae": "Avestan", "af": "Afrikaans", 
                "ak": "Akan", "am": "Amharic", "an": "Aragonese", "ar": "Arabic", 
                "as": "Assamese", "av": "Avaric", "ay": "Aymara", "az": "Azerbaijani", 
                "ba": "Bashkir", "be": "Belarusian", "bg": "Bulgarian", "bh": "Bihari languages", 
                "bi": "Bislama", "bm": "Bambara", "bn": "Bengali", "bo": "Tibetan", "br": "Breton", 
                "bs": "Bosnian", "ca": "Catalan; Valencian", "ce": "Chechen", "ch": "Chamorro", 
                "co": "Corsican", "cr": "Cree", "cs": "Czech", 
                "cu": "Church Slavic; Old Slavonic; Church Slavonic; Old Bulgarian; Old Church Slavonic", 
                "cv": "Chuvash", "cy": "Welsh", "da": "Danish", "de": "German", "dv": "Divehi; Dhivehi; Maldivian", 
                "dz": "Dzongkha", "ee": "Ewe", "el": "Greek, Modern (1453-)", "en": "English", "eo": "Esperanto", 
                "es": "Spanish; Castilian", "et": "Estonian", "eu": "Basque", "fa": "Persian", "ff": "Fulah",
                "fi": "Finnish", "fj": "Fijian", "fo": "Faroese", "fr": "French", "fy": "Western Frisian", 
                "ga": "Irish", "gd": "Gaelic; Scomttish Gaelic", "gl": "Galician", "gn": "Guarani", 
                "gu": "Gujarati", "gv": "Manx", "ha": "Hausa", "he": "Hebrew", "hi": "Hindi", "ho": "Hiri Motu",
                "hr": "Croatian", "ht": "Haitian; Haitian Creole", "hu": "Hungarian", "hy": "Armenian", 
                "hz": "Herero", "ia": "Interlingua (International Auxiliary Language Association)", "id": "Indonesian",
                "ie": "Interlingue; Occidental", "ig": "Igbo", "ii": "Sichuan Yi; Nuosu", "ik": "Inupiaq", "io": "Ido",
                "is": "Icelandic", "it": "Italian", "iu": "Inuktitut", "ja": "Japanese", "jv": "Javanese", "ka": "Georgian",
                "kg": "Kongo", "ki": "Kikuyu; Gikuyu", "kj": "Kuanyama; Kwanyama", "kk": "Kazakh", 
                "kl": "Kalaallisut; Greenlandic", "km": "Central Khmer", "kn": "Kannada", "ko": "Korean",
                "kr": "Kanuri", "ks": "Kashmiri", "ku": "Kurdish", "kv": "Komi", "kw": "Cornish", 
                "ky": "Kirghiz; Kyrgyz", "la": "Latin", "lb": "Luxembourgish; Letzeburgesch", "lg": "Ganda", 
                "li": "Limburgan; Limburger; Limburgish", "ln": "Lingala", "lo": "Lao", "lt": "Lithuanian", 
                "lu": "Luba-Katanga", "lv": "Latvian", "mg": "Malagasy", "mh": "Marshallese", "mi": "Maori",
                "mk": "Macedonian", "ml": "Malayalam", "mn": "Mongolian", "mr": "Marathi", "ms": "Malay",
                "mt": "Maltese", "my": "Burmese", "na": "Nauru", "nb": "Bokmål, Norwegian; Norwegian Bokmål",
                "nd": "Ndebele, North; North Ndebele", "ne": "Nepali", "ng": "Ndonga", "nl": "Dutch; Flemish", 
                "nn": "Norwegian Nynorsk; Nynorsk, Norwegian", "no": "Norwegian", "nr": "Ndebele, South; South Ndebele", 
                "nv": "Navajo; Navaho", "ny": "Chichewa; Chewa; Nyanja", "oc": "Occitan (post 1500)", 
                "oj": "Ojibwa", "om": "Oromo", "or": "Oriya", "os": "Ossetian; Ossetic", "pa": "Panjabi; Punjabi", 
                "pi": "Pali", "pl": "Polish", "ps": "Pushto; Pashto", "pt": "Portuguese", "qu": "Quechua", 
                "rm": "Romansh", "rn": "Rundi", "ro": "Romanian; Moldavian; Moldovan", "ru": "Russian", 
                "rw": "Kinyarwanda", "sa": "Sanskrit", "sc": "Sardinian", "sd": "Sindhi", "se": "Northern Sami",
                "sg": "Sango", "si": "Sinhala; Sinhalese", "sk": "Slovak", "sl": "Slovenian", "sm": "Samoan", 
                "sn": "Shona", "so": "Somali", "sq": "Albanian", "sr": "Serbian", "ss": "Swati", "st": "Sotho, Southern", 
                "su": "Sundanese", "sv": "Swedish", "sw": "Swahili", "ta": "Tamil", "te": "Telugu", "tg": "Tajik", 
                "th": "Thai", "ti": "Tigrinya", "tk": "Turkmen", "tl": "Tagalog", "tn": "Tswana", "to": "Tonga (Tonga Islands)",
                "tr": "Turkish", "ts": "Tsonga", "tt": "Tatar", "tw": "Twi", "ty": "Tahitian", "ug": "Uighur; Uyghur", 
                "uk": "Ukrainian", "ur": "Urdu", "uz": "Uzbek", "ve": "Venda", "vi": "Vietnamese", "vo": "Volapük", 
                "wa": "Walloon", "wo": "Wolof", "xh": "Xhosa", "yi": "Yiddish", "yo": "Yoruba", "za": "Zhuang; Chuang",
                "zh": "Chinese", "zu": "Zulu"
}

def add_CV_data(kg, data, country_map, education_map, fluency_map, language_map):

    cv_results = defaultdict(list)
    
    for candidate in tqdm(data):
        # Clean and gather data
        name = prepare_node(candidate["headline"])
        country = country_map.get(candidate["countryid"], "unknown")
        education = education_map.get(candidate["educationlevel"], "unknown")
    
        cv_id = "candidate_" + candidate["cvid"]

        cv_results["id"].append(candidate["cvid"])

        triples_list = []

        # Triples for df
        triples_list.append((cv_id, "is_a", name))
        triples_list.append((cv_id, "has_education_level", education))
        triples_list.append((cv_id, "number_of_jobs", candidate["jobexperiences"]))
        triples_list.append((cv_id, "managerial_experience", candidate["mgrexperiences"]))
        triples_list.append((cv_id, "lives_in", country))
                             
        # Add triples
        add_triple(kg, ('jie', cv_id), ("rdf", "type"), ('jie', name))
        add_triple(kg, ('jie', cv_id), ("jip", "has_education_level"), ("jie", education))
        add_triple(kg, ('jie', cv_id), ("jip", "number_of_jobs"), candidate["jobexperiences"], literal=True)
        add_triple(kg, ('jie', cv_id), ("jip", "managerial_experience"), candidate["mgrexperiences"], literal=True)
        add_triple(kg, ('jie', cv_id), ("jip", "lives_in"), ("jie", country))
        
        # Add looped triples
        if type(candidate["job_history"]) != float:
            for job in candidate["job_history"]:
                job = prepare_node(job)
                add_triple(kg, ('jie', cv_id), ('jip', "has_worked_position"), ('jie', job))
                
                triples_list.append((cv_id, "has_worked_position", job))
    
        if type(candidate["educations"]) != float:
            for education in candidate["educations"]:
                edu = prepare_node(education["name"])
                if education["type"] == "education":
                    add_triple(kg, ('jie', cv_id), ('jip', "has_education"), ('jie', edu))
                    
                    triples_list.append((cv_id, "has_education", edu))
                elif education["type"] == "course":
                    add_triple(kg, ('jie', cv_id), ('jip', "has_course"), ('jie', edu))

                    triples_list.append((cv_id, "has_course", edu))
                else:
                    pass
    
        if type(candidate["languages"]) != float:
            for language in candidate["languages"]:
                add_triple(kg, ('jie', cv_id), 
                           ('jip', fluency_map.get(language["level"], language["level"])), 
                           ('jie', language_map.get(language["code"], language["code"])))

                triples_list.append((cv_id, 
                                     fluency_map.get(language["level"], language["level"]),
                                     language_map.get(language["code"], language["code"])))
    
        if type(candidate["jobtitles"]) != float:
            for title in candidate["jobtitles"]:
                title = prepare_node(title)
                add_triple(kg, ('jie', cv_id), ('jip', "has_job_title"), ('jie', title))

                triples_list.append((cv_id, "has_job_title", title))
    
        if type(candidate["keywords"]) != float:
            for keyword in candidate["keywords"]:
                keyword = prepare_node(keyword)
                add_triple(kg, ('jie', cv_id), ('jip', "has_keyword"), ('jie', keyword))

                triples_list.append((cv_id, "has_keyword", keyword))

        triples_list = [triple for triple in triples_list if not any([pd.isna(n) for n in triple])]
        cv_results["CV_triples"].append(triples_list)

    df_CVs = pd.DataFrame(cv_results)
    return kg, df_CVs

kg, df_CVs = add_CV_data(kg, data, country_map, education_map, fluency_map, language_map)

  0%|          | 0/83804 [00:00<?, ?it/s]

In [14]:
df_CVs

,id,CV_triples
0,ebf358ed252344af8d1dd3f4c8516cbf,"[(candidate_ebf358ed252344af8d1dd3f4c8516cbf, ..."
1,6c6e7a728dcc4ade98e1367f4efe6fb7,"[(candidate_6c6e7a728dcc4ade98e1367f4efe6fb7, ..."
2,b1afb88436354245a405bcf36f4030c8,"[(candidate_b1afb88436354245a405bcf36f4030c8, ..."
3,e9e192123ed44166993497619c499bd5,"[(candidate_e9e192123ed44166993497619c499bd5, ..."
4,46353e9962684179846f77175feebef9,"[(candidate_46353e9962684179846f77175feebef9, ..."
...,...,...
83799,b2598bc30ed2432cb043a7486e30ced8,"[(candidate_b2598bc30ed2432cb043a7486e30ced8, ..."
83800,dcd0bbd72f23421091ebc5083a77bf10,"[(candidate_dcd0bbd72f23421091ebc5083a77bf10, ..."
83801,aa450872e1c946a78456f91f6c212445,"[(candidate_aa450872e1c946a78456f91f6c212445, ..."
83802,9381173e0e374dc6b60a7719f815ee0e,"[(candidate_9381173e0e374dc6b60a7719f815ee0e, ..."


In [15]:
df_CVs.to_excel("../outputs/clean_outputs/full_cv_triples.xlsx")

In [16]:
measure = kglab.Measure()
measure.measure_graph(kg)

print("edges before inference", measure.get_edge_count())
print("nodes before inference", measure.get_node_count())

# Do all the nifty inference
kg.infer_owlrl_closure()

measure.measure_graph(kg)

print()
print("edges after inference", measure.get_edge_count())
print("nodes after inference", measure.get_node_count())

edges before inference 2205093
nodes before inference 315196

edges after inference 4724346
nodes after inference 315288


In [20]:
### TODO: LOAD ISCO ESCO (both CV and vacancy)
df_isco_cv = pd.DataFrame()

In [40]:
def add_triple(kg, ns_jie, ns_jip, row, triples, triple, set_anchor, kind="vacancy"):
    # Clean all three labels
    i = [clean_label(str(text)).lower() for text in triple]
    # Fix python words
    i = ["c" + text if text == "global" else text for text in i]
    
    # Malformed triple
    if len(i) != 3:
        return kg, None

    # Any of the s, p, o, is empty
    if any([i[0] == "", i[1] == "", i[2] == ""]):
        return kg, None

    # fix digits
    if i[0][0].isdigit():
        i[0] = "_" + i[0]
    
    if i[1][0].isdigit():
        i[1] = "_" + i[1]    
    
    if kind == "vacancy":
        # Add a triple with job_id has_title job_title
        if set_anchor:
            anchor = i[0]
            kg_curr.add(ns_jie[f"position_{triples.loc[row, 'id']}"],
                        kg.get_ns("owl").sameAs,
                        ns_jie[f"{anchor}_{triples.loc[row, 'id']}"])

            # Update node to anchor
            for idx, entity in enumerate(i):
                if entity == anchor:
                    i[idx] = f"{anchor}_{triples.loc[row, 'id']}"
    
        set_anchor = False

    # If s, p, [digit] act accordingly
    if i[2].isdigit():
        kg_curr.add(ns_jie[i[0]], 
                    ns_jip[i[1]], 
                    rdflib.Literal(int(i[2]), datatype=kg_curr.get_ns("xsd").integer))
    elif re.fullmatch(r"\d{4}_\d{2}_\d{2}", i[2]):
        pass

    # More digit cleaning
    if i[2][0].isdigit():
        i[2] = "_" + i[2]

    # Add the actual triple
    kg_curr.add(ns_jie[i[0]], 
                ns_jip[i[1]], 
                ns_jie[i[2]])

    return kg_curr, set_anchor

In [41]:
def add_all_triples(kg_curr, triples, isco_triples, isco_cv_triples, model, prompt):
    print("Adding triples for", model, prompt)

    ns_jie = kg_curr.get_ns('jie')
    ns_jip = kg_curr.get_ns('jip')

    for row, listing in tqdm(enumerate(triples[f"triples_{model}_{prompt}"])):
        extracted_triples = extract_triples(listing)

        set_anchor = True
        
        for triple in extracted_triples:
            kg_curr, set_anchor = add_triple(kg_curr, ns_jie, ns_jip, 
                                             row, triples, triple, set_anchor, kind="vacancy")

    for row, vacancy in tqdm(enumerate(isco_triples.iterrows())):
        isco = extract_triples(vacancy[1][f"ISCO {model} {prompt}"])
        esco = extract_triples(vacancy[1][f"ESCO {model} {prompt}"])

        for triple in isco:
            kg_curr, set_anchor = add_triple(kg_curr, ns_jie, ns_jip, 
                                             row, triples, triple, set_anchor, kind="isco")

        for triple in esco:
            kg_curr, set_anchor = add_triple(kg_curr, ns_jie, ns_jip, 
                                             row, triples, triple, set_anchor, kind="esco")

    for row in df_isco_cv:
        pass

    before = time.time()
    
    measure = kglab.Measure()
    measure.measure_graph(kg_curr)
    
    print("edges before inference", measure.get_edge_count())
    print("nodes before inference", measure.get_node_count())
    
    # Do all the nifty inference
    kg_curr.infer_owlrl_closure()
    
    measure.measure_graph(kg_curr)
    
    print()
    print("edges after inference", measure.get_edge_count())
    print("nodes after inference", measure.get_node_count())

    print(f"Inference took {np.round(time.time() - before, 2)} seconds")
    
    # create a Tensor representation (replaces SubgraphTensor)
    kgt = kg_curr.rdf_graph()
    
    with open(f"../outputs/inferred_outputs/kg_{model}_{prompt}.edgelist", "w+", encoding="utf-8") as f:
        
        for i, (s, p, o) in tqdm(enumerate(kgt), total=len(kgt)):
                s_label = kg_curr.n3fy(s)
                p_label = kg_curr.n3fy(p)
                o_label = kg_curr.n3fy(o)
    
                f.write(f"['{s_label}', '{p_label}', '{o_label}']\n")

for model in ["qwen", "gemma", "llama"]:
    for prompt in ["structured", "semi-structured", "unstructured"]:  
        print("Copying kg")
        # kg_curr = copy.deepcopy(kg)
        add_all_triples(kg_curr, df, df_isco, df_isco_cv, model, prompt)

Copying kg
Adding triples for qwen structured


0it [00:00, ?it/s]

0it [00:00, ?it/s]

edges before inference 2693224
nodes before inference 381820

edges after inference 5479298
nodes after inference 383577
Inference took 1370.51 seconds


  0%|          | 0/2786074 [00:00<?, ?it/s]

Copying kg
Adding triples for qwen semi-structured


0it [00:00, ?it/s]

0it [00:00, ?it/s]

edges before inference 2927630
nodes before inference 412883


KeyboardInterrupt: 

## Descriptives

In [ ]:
def graph_info(G):
    info = [
        f"Name: {G.name}",
        f"Type: {type(G).__name__}",
        f"Nodes: {G.number_of_nodes()}",
        f"Edges: {G.number_of_edges()}",
        f"Is directed: {G.is_directed()}",
    ]
    return "\n".join(info)

In [ ]:
# Run your SPARQL query
sparql = """
SELECT ?subject ?object WHERE {
  ?subject ?pred ?object .
}
"""
results = kg.query(sparql)

# Build a NetworkX graph manually
G = nx.DiGraph()

for row in results:
    subject = str(row.subject)
    obj = str(row.object)
    G.add_edge(subject, obj)

print(graph_info(G))

In [ ]:
np.mean(list(dict(G.degree()).values()))

In [ ]:
# Access the underlying rdflib graph
graph = kg.rdf_graph()

# Extract predicates
predicates = [str(p) for (_, p, _) in graph]

# Count occurrences
predicate_counts = Counter(predicates)

len(predicate_counts), predicate_counts

In [ ]:
# create a Tensor representation (replaces SubgraphTensor)
kgt = kg.rdf_graph()

with open(f"../outputs/inferred_outputs/kg_{model}_{prompt}.edgelist", "w+", encoding="utf-8") as f:
    
    for i, (s, p, o) in tqdm(enumerate(kgt), total=len(kgt)):
            s_label = kg.n3fy(s)
            p_label = kg.n3fy(p)
            o_label = kg.n3fy(o)

            f.write(f"['{s_label}', '{p_label}', '{o_label}']\n")